---
# LLPS-ScanAI  
Welcome to **LLPS-ScanAI**, a user-friendly notebook to predict phase-separating driver regions in proteins using protein language models and deep learning.
- **Input**: an UniProt ID or a protein sequence.
- **Output**: LLPS propensity profile table and figure.  
- **How?** Just go to `Runtime` → `Run all` or press `ctrl+F9`  
- **Results** will be displayed at the bottom of this notebook.  

---

# 1. Set up

In [ ]:
#@title Input protein data { display-mode: "form", form-width: "100%" }

#@markdown - Enter an UniProt ID in the box below.
uniprot_id = "D0PV95"  #@param {type:"string"}
#@markdown - Or input a protein sequence directamente.
input_sequence = ""  #@param {type:"string"}

#@markdown You can try the example sequence provided below:
use_example= False  #@param {type:"boolean"}

if uniprot_id or input_sequence:
    use_example = False 

if use_example:
    uniprot_id = "D0PV95" 
    input_sequence = "MDVFMKGLSKAKEGVVAAAEKTKQGVAEAAGKTKEGVLYVGSKTKEGVVHGVATVAEKTKEQVTNVGGAVVTGVTAVAQKTVEGAGSIAAATGFVKKDQLGKNEEGAPQEGILEDMPVDPDNEAYEMPSEEGYQDYEPEA"

#@markdown ---
import time
start_time = time.time()

In [ ]:
#@title Retrieve sequence from UniProt if needed { display-mode: "form", form-width: "100%"  }
import requests
import pandas as pd

if not use_example and uniprot_id:
    uniprot_id = uniprot_id.strip() 
    url = f"https://rest.uniprot.org/uniprotkb/{uniprot_id}.fasta"
    response = requests.get(url)
    if response.status_code == 200:
        fasta_lines = response.text.strip().split('\n')
        input_sequence = ''.join(fasta_lines[1:])
    else:
        raise ValueError(f"Failed to fetch sequence for UniProt ID {uniprot_id}.")

input_sequence = input_sequence.replace(' ', '').replace('\n', '').upper() 
if input_sequence == "":
    raise ValueError("⚠️ No sequence provided.")

df = pd.DataFrame({'uniprot_id': [uniprot_id], 'sequence': [input_sequence]})

In [ ]:
#@title Download model from GitLab { display-mode: "form", form-width: "100%" }
import os
import urllib.request
from tqdm import tqdm

os.makedirs("models", exist_ok=True)
base_url = "https://gitlab.com/azourbarbar/llps-scanai/-/raw/master/models"

model_files = [f"model_fold_{i}.pth" for i in range(1, 6)] + ["model_config.json"]

print("Downloading LLPS-ScanAI models...")
for fname in tqdm(model_files):
    model_url = f"{base_url}/{fname}"
    model_path = f"models/{fname}"
    if not os.path.exists(model_path):
        urllib.request.urlretrieve(model_url, model_path)

# 2. Predict

In [ ]:
%%capture
#@title Load models { display-mode: "form", form-width: "100%" }
import torch
from torch import nn
import json

# Arquitectura MLP exacta de tu entrenamiento
class MLP(nn.Module):
    def __init__(self, in_dim=1024, p=0.3):
        super().__init__()
        self.net = nn.Sequential(
            nn.Linear(in_dim, 512), nn.BatchNorm1d(512), nn.ReLU(), nn.Dropout(p),
            nn.Linear(512, 256), nn.BatchNorm1d(256), nn.ReLU(), nn.Dropout(p),
            nn.Linear(256, 128), nn.BatchNorm1d(128), nn.ReLU(), nn.Dropout(p),
            nn.Linear(128, 64),  nn.BatchNorm1d(64),  nn.ReLU(), nn.Dropout(p),
            nn.Linear(64, 1)
        )
    def forward(self, x): return self.net(x)

device = torch.device("cuda" if torch.cuda.is_available() else "cpu")

with open('models/model_config.json', 'r') as f:
    config = json.load(f)
best_thr = config.get("best_threshold", 0.17)

llps_models = []
for i in range(1, 6):
    m = MLP(1024).to(device)
    m.load_state_dict(torch.load(f"models/model_fold_{i}.pth", map_location=device))
    m.eval()
    llps_models.append(m)

In [ ]:
#@title Generate embedding representations { display-mode: "form", form-width: "100%" }
import torch
from transformers import T5Tokenizer, T5EncoderModel
from tqdm import tqdm

transformer_link = "Rostlab/prot_t5_xl_half_uniref50-enc"
device = torch.device("cuda:0" if torch.cuda.is_available() else "cpu")
model = T5EncoderModel.from_pretrained(transformer_link).to(device).eval()
tokenizer = T5Tokenizer.from_pretrained(transformer_link, do_lower_case=False, legacy=False)

def generate_embeddings(sequence: str):
    spaced = " ".join(list(sequence))
    ids = tokenizer(spaced, add_special_tokens=True, return_tensors="pt").to(device)
    with torch.no_grad():
        out = model(input_ids=ids["input_ids"], attention_mask=ids["attention_mask"])
    return out.last_hidden_state[0, :-1].cpu().numpy()

tqdm.pandas(desc="Generating embeddings")
df["embedding"] = df["sequence"].progress_map(generate_embeddings)

In [ ]:
#@title Run Predictions { display-mode: "form", form-width: "100%" }
import numpy as np

def ensemble_predict(embedding):
    X = torch.tensor(embedding).float().to(device)
    with torch.no_grad():
        # Predicción promediada de los 5 modelos
        all_logits = [m(X).squeeze(1).cpu().numpy() for m in llps_models]
    avg_logits = np.mean(all_logits, axis=0)
    # Aplicamos Sigmoid manual ya que el modelo devuelve Logits
    probs = 1 / (1 + np.exp(-avg_logits))
    return probs

tqdm.pandas(desc="Computing probabilities")
df["prob_vector"] = df["embedding"].progress_map(ensemble_predict)

# 3. Results

In [ ]:
#@title Visualize results { display-mode: "form", form-width: "100%" }
import plotly.graph_objects as go
from plotly.subplots import make_subplots

window_size = 10  #@param {type:"slider", min:1, max:25, step:1}

def moving_average(x, w=5):
    return np.convolve(x, np.ones(w)/w, mode='same')

row = df.iloc[0]
residues = list(row["sequence"])
positions = np.arange(1, len(residues) + 1)
prob_vector = np.array(row["prob_vector"])
smoothed = moving_average(prob_vector, w=window_size)

result_df = pd.DataFrame({
    "uniprot_id": [row["uniprot_id"]] * len(positions),
    "position": positions,
    "residue": residues,
    "llps_score": prob_vector
})

fig_only = go.Figure()
fig_only.add_trace(go.Scatter(x=positions, y=prob_vector, fill='tozeroy', mode='lines', line=dict(color='lightgray'), name='Raw Score'))
fig_only.add_trace(go.Scatter(x=positions, y=smoothed, mode='lines+markers', name='Smoothed (Window)', line=dict(color='blue', width=2), marker=dict(size=4), text=residues))
fig_only.add_hline(y=best_thr, line_dash="dash", line_color="red", annotation_text=f"Threshold {best_thr:.3f}")

fig_only.update_layout(title=f"LLPS Propensity Profile ({row.uniprot_id})", xaxis_title='Residue Position', yaxis_title='LLPS Propensity', template='simple_white', height=700)
fig_only.show()

In [ ]:
#@title Download results { display-mode: "form", form-width: "100%" }
from google.colab import files

filename = f"llpsscanai_results_{uniprot_id}"
result_df.to_csv(f"{filename}.csv", index=False)
fig_only.write_html(f"{filename}.html")

files.download(f"{filename}.csv")
files.download(f"{filename}.html")

end_time = time.time()
print(f"Total execution time: {(end_time - start_time)/60:.2f} minutes")